<a href="https://colab.research.google.com/github/peremartra/fairness-pruning/blob/main/notebooks/02_Evaluate_BBQ_Instruct.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fairness Pruning Research - Instruct Model Evaluation
## 02 - BBQ with explicit options (A/B/C) and chat template

### Establishing Performance Bias using BBVQ, EsBBQ & CatBBQ for Bias Mitigation Research
by [Pere Martra](https://github.com/peremartra)

[![GitHub](https://img.shields.io/badge/⭐_Star-OptiPFair-orange?logo=github&logoColor=white)](https://github.com/peremartra/optipfair)
[![PyPI](https://img.shields.io/pypi/v/optipfair?logo=python&logoColor=white&label=v)](https://pypi.org/project/optipfair/)

**Repository:** [github.com/peremartra/fairness-pruning](https://github.com/peremartra/fairness-pruning)

---

**Colab Environment:** GPU L4 or A100

**Models to Evaluate:**
* Llama-3.2-3B-Instruct

---

**Note:** This notebook uses the custom task `bbq_instruct_mc` (`custom_tasks/bbq_instruct/`), not lm-eval's default `bbq`. Its numbers are not comparable with the base-model results of `02_Evaluate_BBQ.ipynb`.

---

# 02 — BBQ evaluation for instruct models (options A/B/C, chat template)

## Format

- Task: `bbq_instruct_mc` (see `custom_tasks/bbq_instruct/README.md`).
- Prompt: neutral instruction + context + question + options A/B/C in the dataset order, sent as the user message through the model's chat template (`apply_chat_template=True`). No system prompt. The instruction does not mention when to answer "unknown".
- Scoring: log-likelihood of the letter; 0-shot.
- Dataset and metric code are the same as lm-eval's default `bbq` (imported from `lm_eval.tasks.bbq.utils`).

## Evaluation log

| Run | Date | Models | Task | Settings |
|---|---|---|---|---|
| 1 | [date] | meta-llama/Llama-3.2-3B-Instruct | `bbq_instruct_mc` (English, full dataset) | 0-shot, chat template, bfloat16, `batch_size=16`, versions and GPU as printed by the setup cell |

Notes:
- Checkpoints: `fair_pruning/checkpoints_bbq_instruct/` (separate from the base-model checkpoints).
- Per-item predictions: `fair_pruning/checkpoints_bbq_instruct/results/lm_evals/<model>_bbq_instruct_mc.json`.
- Consolidated files: `fair_pruning/results/instruct_models_bbq_mc_results_*.csv/json` (they do not overwrite the base-model `base_models_bbq_results_latest.csv`).

# 1. Setup & Installation

In [ ]:
# Clone the repository (light: no history, only notebooks and custom tasks)
import os, subprocess, sys

REPO_URL = "https://github.com/peremartra/fairness-pruning.git"
REPO_DIR = "fairness-pruning"

if not os.path.exists(REPO_DIR):
    !git clone -q --depth 1 --filter=blob:none --sparse {REPO_URL}
    !cd {REPO_DIR} && git sparse-checkout set notebooks custom_tasks
else:
    print("Repo already cloned. Pulling latest changes...")
    !cd {REPO_DIR} && git pull -q

%cd {REPO_DIR}
print(f"Working directory: {os.getcwd()}")

# Install dependencies from requirements.txt
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-r", "requirements.txt", ""]
)
print("Dependencies installed.")

# Versions actually in use (for the evaluation log)
from importlib.metadata import version
for pkg in ("torch", "transformers", "lm-eval", "accelerate", "datasets"):
    print(f"{pkg:14s} {version(pkg)}")
print("Python:", sys.version.split()[0])
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# Mount Google Drive for checkpoint persistence
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Register the custom task bbq_instruct_mc in lm-eval
import os, shutil
import lm_eval

TASK_SRC = "custom_tasks/bbq_instruct"
TASK_DST = os.path.join(os.path.dirname(lm_eval.__file__), "tasks", "bbq_instruct")

shutil.rmtree(TASK_DST, ignore_errors=True)
shutil.copytree(TASK_SRC, TASK_DST)
print(f"Task installed in: {TASK_DST}")
print("Files:", sorted(os.listdir(TASK_DST)))

# 1. Helper Functions

Utility functions for automatic checkpoint path generation and model size detection.

# 2. Configuration & Evaluation Plan (BBQ instruct, options A/B/C)

Configure paths, select the `bbq_instruct_mc` task, and list the instruct models we will evaluate.

In [ ]:
import torch
import json
import pandas as pd
from datetime import datetime
from pathlib import Path
import logging
import os
from transformers import AutoModelForCausalLM, AutoTokenizer
from IPython.display import display
from utils import (
    #EXPERIMENT_CONFIG,
    run_robust_evaluation,
    load_or_create_model,
    clear_gpu_cache,
    format_results_table,
    get_model_stats,
 )

# Paths (Drive recommended in Colab)
CHECKPOINT_BASE_DIR = "/content/drive/MyDrive/fair_pruning/checkpoints_bbq_instruct"
RESULTS_DIR = "/content/drive/MyDrive/fair_pruning/results"
Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)

# BBQ task list (0-shot, custom task with options A/B/C)
BBQ_TASKS = [{"name": "bbq_instruct_mc", "num_fewshot": 0}]

# De-duplicate models from EXPERIMENT_CONFIG
EXPERIMENT_CONFIG = [
    {
        "base_model": "meta-llama/Llama-3.2-3B-Instruct",
    },
]
unique_models = list(dict.fromkeys([cfg["base_model"] for cfg in EXPERIMENT_CONFIG]))

logging.getLogger("lm_eval").setLevel(logging.INFO)

print(f"{'='*70}")
print("📊 EVALUATION PLAN: BBQ instruct (options A/B/C, chat template)")
print(f"{'='*70}")
print(f"Checkpoints: {CHECKPOINT_BASE_DIR}")
print(f"Results: {RESULTS_DIR}")
print(f"Tasks: {[task['name'] for task in BBQ_TASKS]}")
print("Models:")
for m in unique_models:
    print(f" - {m}")
print(f"{'='*70}\n")

In [ ]:
import re
import os

def get_model_size(model_name: str) -> str:
    """Extract model size identifier from HuggingFace model name.

    Examples:
        "meta-llama/Llama-3.2-1B" → "1b"
        "meta-llama/Llama-3.2-3B-Instruct" → "3b_instruct"
        "BSC-LT/salamandra-2b" → "2b"
    """
    match = re.search(r'(\d+\.?\d*)[Bb]', model_name)
    if not match:
        return "unknown"

    size = match.group(1).replace('.', '_') + "b"
    if "instruct" in model_name.lower():
        size += "_instruct"

    return size.lower()

def get_checkpoint_path(model_name: str, base_dir: str) -> str:
    """Generate checkpoint path with size-based subdirectory.

    Args:
        model_name: Full HuggingFace model identifier
        base_dir: Base directory for checkpoints

    Returns:
        Full path to checkpoint file
    """
    model_size = get_model_size(model_name)
    safe_name = model_name.replace('/', '_').replace('-', '_').lower()
    checkpoint_dir = os.path.join(base_dir, model_size)
    os.makedirs(checkpoint_dir, exist_ok=True)
    return os.path.join(checkpoint_dir, f"{safe_name}.json")

# Test with EXPERIMENT_CONFIG
print("Testing helper functions with EXPERIMENT_CONFIG:")
print("-" * 70)
for cfg in EXPERIMENT_CONFIG:
    model_id = cfg['base_model']
    size = get_model_size(model_id)
    print(f"{model_id:<50} → {size}")
print("-" * 70)

# 3. Run BBQ Evaluation

Evaluate each instruct model on `bbq_instruct_mc` (chat template) with checkpoint/resume and raw result saving.

In [ ]:
print(f"\n{'='*70}")
print("🚀 STARTING BBQ EVALUATION (Hugging Face Native Loading)")
print(f"{'='*70}\n")

all_model_results = {}

for idx, model_id in enumerate(unique_models, 1):
    print(f"\n{'='*70}")
    print(f"📊 MODEL {idx}/{len(unique_models)}: {model_id}")
    print(f"{'='*70}\n")
    try:
        # Generar ruta de checkpoint
        checkpoint_path = get_checkpoint_path(model_id, CHECKPOINT_BASE_DIR)

        # --- CARGA DIRECTA CON HUGGING FACE ---
        print(f"📥 Loading directly from Hugging Face Hub...")
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            torch_dtype=torch.bfloat16,
            device_map="auto"
        )

        tokenizer = AutoTokenizer.from_pretrained(model_id)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        assert tokenizer.chat_template, f"{model_id} has no chat template"

        print("✅ Model loaded successfully (Native HF)")

        # Mostrar estadísticas
        stats = get_model_stats(model)
        print(f"📈 Params: {stats['total_parameters']:,} | Size: {stats['size_gb']:.2f} GB")
        print(f"📁 Checkpoint: {checkpoint_path}\n")

        # --- EJECUTAR EVALUACIÓN (Llamada corregida) ---
        results = run_robust_evaluation(
            model=model,
            tokenizer=tokenizer,
            tasks=BBQ_TASKS,
            checkpoint_path=checkpoint_path,
            model_name=model_id,
            batch_size=16,
            apply_chat_template=True
        )
        # -----------------------------------------------

        all_model_results[model_id] = results
        print(f"\n✅ Completed: {model_id}")
        print("Results Preview:")
        print(format_results_table(results))

    except Exception as e:
        print(f"\n❌ ERROR evaluating {model_id}: {e}")
        if 'model' in locals(): del model
        if 'tokenizer' in locals(): del tokenizer
        clear_gpu_cache()
        continue

    # Limpieza de memoria
    del model
    del tokenizer
    clear_gpu_cache()

print(f"\n{'='*70}")
print(f"✅ BBQ EVALUATION COMPLETE: {len(all_model_results)}/{len(unique_models)} models")
print(f"{'='*70}\n")

# 4. Consolidate BBQ Results

Load checkpoint files, flatten metrics, and export combined BBQ results.

In [ ]:
import glob

def flatten_metrics(metrics, prefix=''):
    flat = {}
    for k, v in metrics.items():
        if isinstance(v, dict):
            flat.update(flatten_metrics(v, prefix=f"{prefix}{k}_"))
        else:
            flat[f"{prefix}{k}"] = v
    return flat

print(f"{'='*70}")
print(" CONSOLIDATING BBQ RESULTS")
print(f"{'='*70}\n")

checkpoint_files = [
    f for f in glob.glob(f"{CHECKPOINT_BASE_DIR}/**/*.json", recursive=True)
    if "lm_evals" not in f
]
print(f"Total checkpoint JSONs: {len(checkpoint_files)}")

consolidated_data = []

for json_path in sorted(checkpoint_files):
    if "lm_evals" in json_path:
        continue  # skip raw per-task dumps
    try:
        with open(json_path, "r") as f:
            data = json.load(f)

        metadata = data.get("metadata", {})
        model_name = metadata.get("model_name", "Unknown")

        model_size = get_model_size(model_name)

        results = data.get("results", {})
        if not results:
            continue

        for task_name, metrics in results.items():
            row = {
                "model": model_name,
                "model_size": model_size,
                "task": task_name,
            }
            row.update(flatten_metrics(metrics))
            if "accuracy" not in row and "acc" in row:
                row["accuracy"] = row["acc"]
            consolidated_data.append(row)

    except Exception as e:
        print(f"   -> Error reading {json_path}: {e}")

if consolidated_data:
    df = pd.DataFrame(consolidated_data)
    df = df.sort_values(by=["model", "task"]).reset_index(drop=True)
    display(df.head())

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    csv_path = f"{RESULTS_DIR}/instruct_models_bbq_mc_results_{timestamp}.csv"
    latest_csv = f"{RESULTS_DIR}/instruct_models_bbq_mc_results_latest.csv"
    json_path = f"{RESULTS_DIR}/instruct_models_bbq_mc_results_{timestamp}.json"

    df.to_csv(csv_path, index=False)
    df.to_csv(latest_csv, index=False)
    df.to_json(json_path, orient="records", indent=2)

    print("\n💾 BBQ results saved:")
    print(f"   {csv_path}")
    print(f"   {latest_csv}")
    print(f"   {json_path}")
else:
    print("⚠️ No BBQ results found to consolidate.")

# 5. Summary Analysis

Quick per-model accuracy summary for BBQ and file inventory.

In [ ]:
# Summary stats
summary_df = None
if 'df' in locals() and not df.empty:
    summaries = []
    for model_name, model_df in df.groupby('model'):
        acc_series = pd.to_numeric(model_df.get('accuracy'), errors='coerce').dropna()
        summaries.append({
            "model": model_name,
            "model_size": model_df.get('model_size', pd.Series(['unknown'])).iloc[0],
            "avg_accuracy": acc_series.mean() if len(acc_series) else None,
            "tasks_completed": len(model_df)
        })

    summary_df = pd.DataFrame(summaries)
    summary_df = summary_df.sort_values("model").reset_index(drop=True)
    print(f"{'='*70}")
    print("📈 BBQ SUMMARY")
    print(f"{'='*70}")
    print(summary_df.to_string(index=False, float_format="%.4f"))

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    summary_csv = f"{RESULTS_DIR}/instruct_models_bbq_mc_summary_{timestamp}.csv"
    summary_df.to_csv(summary_csv, index=False)
    print(f"\n💾 Summary saved: {summary_csv}\n")
else:
    print("No consolidated data available for summary.")

print(f"{'='*70}")
print("📁 GENERATED FILES")
print(f"{'='*70}")

if 'csv_path' in locals() and os.path.exists(csv_path):
    print(f"  ✅ {csv_path}")
if 'latest_csv' in locals() and os.path.exists(latest_csv):
    print(f"  ✅ {latest_csv}")
if 'json_path' in locals() and os.path.exists(json_path):
    print(f"  ✅ {json_path}")
if 'summary_csv' in locals() and os.path.exists(summary_csv):
    print(f"  ✅ {summary_csv}")

print("\nCheckpoints (first 10):")
if 'checkpoint_files' in locals():
    for f in sorted(checkpoint_files)[:10]:
        print(f"  ✅ {f}")
    if len(checkpoint_files) > 10:
        print(f"  ... and {len(checkpoint_files) - 10} more")

print(f"{'='*70}")